# Lab 4: TinyML Lab, Quantization-Aware Training (QAT)

**Name:** Tanmay Gautam  
**Register No.:** 2548559  
**Class:** 5MSAIM

## Aim
To implement Quantization-Aware Training (QAT) on a TinyML model using a standard image dataset with MobileNetV2, and to compare it with Post-Training Quantization (PTQ).

**Input data option chosen:** Option B (CIFAR-10 image data with MobileNetV2, alpha = 0.35)

**Abbreviations used**
- QAT: Quantization-Aware Training
- PTQ: Post-Training Quantization
- TFLite: TensorFlow Lite
- INT8: 8-bit integer
- CIFAR-10: Canadian Institute For Advanced Research, 10-class image dataset

## Planned Workflow

1. **Setup:** Install `tensorflow-model-optimization` and `tf_keras`, and turn on legacy Keras.
2. **Load data:** Load CIFAR-10 and take a random subset of 5,000 training and 1,000 test images.
3. **Preprocess:** Resize images from 32 x 32 to 96 x 96 and scale pixels to the range [-1, 1] using the MobileNetV2 `preprocess_input` function.
4. **Baseline model:** Build MobileNetV2 (pre-trained on ImageNet, alpha = 0.35) with a new classification head as a single functional model (not nested).
5. **Fine-tune:** Phase 1 trains only the new head. Phase 2 unfreezes the whole network with a small learning rate. Record test accuracy.
6. **Float32 TFLite:** Convert the baseline model to a float32 TFLite model.
7. **PTQ:** Convert the baseline model to a full INT8 TFLite model using a representative dataset (200 training images).
8. **QAT:** Wrap the trained baseline with fake-quantization layers, fine-tune for a few epochs, and convert to a full INT8 TFLite model (no representative dataset).
9. **Evaluate:** Run all three `.tflite` models with the TFLite interpreter on the same 1,000 test images. Measure size (KB), accuracy (%) and average latency (ms).
10. **Compare and conclude:** Fill the comparison table, answer the questions, and state the result.

## Planned Workflow

```
Setup (tf_keras, tensorflow-model-optimization)
        |
Load CIFAR-10 subset (5,000 train / 1,000 test)
        |
Preprocess (resize to 96 x 96, scale to [-1, 1])
        |
Build MobileNetV2 (alpha = 0.35) + new head
        |
Fine-tune baseline (head only, then full network)
        |
        +--> Float32 TFLite (plain conversion)
        |
        +--> PTQ full INT8 (with representative dataset)
        |
        +--> QAT (fake-quantization fine-tuning)
                --> full INT8 TFLite (no representative dataset)
        |
Evaluate all 3 models (size KB, accuracy %, latency ms)
        |
Compare in table and write result
```

## 1. Setup

In [1]:
!pip install -q tensorflow-model-optimization tf_keras
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # set before importing tensorflow
import tensorflow as tf
import tensorflow_model_optimization as tfmot

import time
import shutil
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)
print("TF Model Optimization version:", tfmot.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

tf.keras.utils.set_random_seed(42)

TensorFlow version: 2.21.0
TF Model Optimization version: 0.8.1
GPU available: []


In [4]:
%pip install -q certifi
import ssl
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

Note: you may need to restart the kernel to use updated packages.


## 2. Load CIFAR-10 and take a subset

In [7]:
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.cifar10.load_data()

rng = np.random.default_rng(42)
train_idx = rng.choice(len(x_train_full), 5000, replace=False)
test_idx = rng.choice(len(x_test_full), 1000, replace=False)

x_train_raw = x_train_full[train_idx]
y_train = y_train_full[train_idx].flatten()
x_test_raw = x_test_full[test_idx]
y_test = y_test_full[test_idx].flatten()

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print("Train subset:", x_train_raw.shape, y_train.shape)
print("Test subset: ", x_test_raw.shape, y_test.shape)
print("Train images per class:", np.bincount(y_train))
print("Test images per class: ", np.bincount(y_test))

Train subset: (5000, 32, 32, 3) (5000,)
Test subset:  (1000, 32, 32, 3) (1000,)
Train images per class: [489 505 521 505 476 495 496 497 510 506]
Test images per class:  [ 93  82 121  97  97 100  97 109 109  95]


## 3. Preprocessing (resize to 96 x 96 and scale for MobileNetV2)

In [8]:
IMG_SIZE = 96
NUM_CLASSES = 10

def preprocess(images, batch=500):
    out = []
    for i in range(0, len(images), batch):
        b = tf.image.resize(tf.cast(images[i:i + batch], tf.float32), (IMG_SIZE, IMG_SIZE))
        b = tf.keras.applications.mobilenet_v2.preprocess_input(b)
        out.append(b.numpy())
    return np.concatenate(out).astype(np.float32)

x_train = preprocess(x_train_raw)
x_test = preprocess(x_test_raw)

print("x_train:", x_train.shape, "range:", x_train.min(), "to", x_train.max())
print("x_test: ", x_test.shape, "range:", x_test.min(), "to", x_test.max())

x_train: (5000, 96, 96, 3) range: -1.0 to 1.0
x_test:  (1000, 96, 96, 3) range: -1.0 to 1.0


## 4. Baseline model: MobileNetV2 (alpha = 0.35) with a new head, as one functional model

In [9]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    alpha=0.35,
    include_top=False,
    weights="imagenet",
)

x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(base.output)
x = tf.keras.layers.Dropout(0.2, name="dropout")(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, name="logits")(x)

model = tf.keras.Model(inputs=base.input, outputs=outputs, name="mobilenetv2_cifar10")

nested = [l.name for l in model.layers if isinstance(l, tf.keras.Model)]
print("Nested sub-models (should be empty):", nested)
print("Number of layers:", len(model.layers))
print("Total parameters:", model.count_params())

LOSS = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
HEAD_LAYERS = {"gap", "dropout", "logits"}

2019640/2019640 [==============================] - 0s 0us/step
Nested sub-models (should be empty): []
Number of layers: 157
Total parameters: 423018


## 5. Fine-tune the baseline
Phase 1 trains only the new head. Phase 2 unfreezes the whole network with a small learning rate.

In [10]:
for layer in model.layers:
    layer.trainable = layer.name in HEAD_LAYERS

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=LOSS, metrics=["accuracy"])
history1 = model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1, verbose=2)

Epoch 1/5
71/71 - 3s - loss: 1.3585 - accuracy: 0.5327 - val_loss: 0.8621 - val_accuracy: 0.7140 - 3s/epoch - 39ms/step
Epoch 2/5
71/71 - 1s - loss: 0.7907 - accuracy: 0.7256 - val_loss: 0.7265 - val_accuracy: 0.7480 - 1s/epoch - 19ms/step
Epoch 3/5
71/71 - 1s - loss: 0.6510 - accuracy: 0.7787 - val_loss: 0.6857 - val_accuracy: 0.7600 - 1s/epoch - 20ms/step
Epoch 4/5
71/71 - 1s - loss: 0.5868 - accuracy: 0.7993 - val_loss: 0.6484 - val_accuracy: 0.7740 - 1s/epoch - 20ms/step
Epoch 5/5
71/71 - 1s - loss: 0.5218 - accuracy: 0.8196 - val_loss: 0.6474 - val_accuracy: 0.7740 - 1s/epoch - 20ms/step


In [11]:
for layer in model.layers:
    layer.trainable = True

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=LOSS, metrics=["accuracy"])
history2 = model.fit(x_train, y_train, epochs=10, batch_size=64, validation_split=0.1,
                     callbacks=[early_stop], verbose=2)

_, keras_baseline_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Keras float32 baseline test accuracy: {keras_baseline_acc * 100:.2f}%")

Epoch 1/10
71/71 - 12s - loss: 0.6923 - accuracy: 0.7669 - val_loss: 0.8877 - val_accuracy: 0.6960 - 12s/epoch - 167ms/step
Epoch 2/10
71/71 - 7s - loss: 0.4331 - accuracy: 0.8520 - val_loss: 0.8307 - val_accuracy: 0.7040 - 7s/epoch - 99ms/step
Epoch 3/10
71/71 - 7s - loss: 0.3097 - accuracy: 0.8978 - val_loss: 0.8097 - val_accuracy: 0.7040 - 7s/epoch - 93ms/step
Epoch 4/10
71/71 - 7s - loss: 0.2384 - accuracy: 0.9233 - val_loss: 0.7925 - val_accuracy: 0.7120 - 7s/epoch - 92ms/step
Epoch 5/10
71/71 - 7s - loss: 0.1800 - accuracy: 0.9476 - val_loss: 0.7932 - val_accuracy: 0.7220 - 7s/epoch - 92ms/step
Epoch 6/10
71/71 - 7s - loss: 0.1458 - accuracy: 0.9589 - val_loss: 0.7884 - val_accuracy: 0.7240 - 7s/epoch - 92ms/step
Epoch 7/10
71/71 - 7s - loss: 0.1182 - accuracy: 0.9736 - val_loss: 0.7763 - val_accuracy: 0.7280 - 7s/epoch - 92ms/step
Epoch 8/10
71/71 - 7s - loss: 0.0957 - accuracy: 0.9778 - val_loss: 0.7564 - val_accuracy: 0.7340 - 7s/epoch - 92ms/step
Epoch 9/10
71/71 - 7s - loss:

## 6. Convert the baseline to float32 TFLite and to full INT8 TFLite (PTQ)

In [12]:
os.makedirs("tflite_models", exist_ok=True)
FLOAT_PATH = "tflite_models/baseline_float32.tflite"
PTQ_PATH = "tflite_models/ptq_int8.tflite"
QAT_PATH = "tflite_models/qat_int8.tflite"

converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite = converter.convert()
with open(FLOAT_PATH, "wb") as f:
    f.write(float_tflite)
print("Saved", FLOAT_PATH)

def representative_dataset():
    for i in range(200):
        yield [x_train[i:i + 1]]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
ptq_tflite = converter.convert()
with open(PTQ_PATH, "wb") as f:
    f.write(ptq_tflite)
print("Saved", PTQ_PATH)

INFO:tensorflow:Assets written to: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmpe2ek9h6_/assets


INFO:tensorflow:Assets written to: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmpe2ek9h6_/assets
W0000 00:00:1791003948.204875 4204729 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1791003948.204889 4204729 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1791003948.205125 4204729 reader.cc:83] Reading SavedModel from: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmpe2ek9h6_
I0000 00:00:1791003948.216822 4204729 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1791003948.216827 4204729 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmpe2ek9h6_
I0000 00:00:1791003948.277412 4204729 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1791003948.286784 4204729 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1791003948.630983 4204729 loader.cc:220] Running initialization op on SavedModel bundle at path

Saved tflite_models/baseline_float32.tflite
INFO:tensorflow:Assets written to: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmpksnd1biy/assets


INFO:tensorflow:Assets written to: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmpksnd1biy/assets
/Users/Tanmay/Downloads/tf_env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1791003959.483884 4204729 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1791003959.483896 4204729 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1791003959.484036 4204729 reader.cc:83] Reading SavedModel from: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmpksnd1biy
I0000 00:00:1791003959.496416 4204729 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1791003959.496422 4204729 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmpksnd1biy
I0000 00:00:1791003959.567479 4204729 loader.cc:236] Restoring SavedModel bundle.
I0000 00:

Saved tflite_models/ptq_int8.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1791003962.544149 4204729 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.


## 7. TFLite evaluation helper (accuracy and latency)

In [14]:
def evaluate_tflite(path, x, y, num_threads=1):
    interpreter = tf.lite.Interpreter(model_path=path, num_threads=num_threads)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    in_scale, in_zero = inp["quantization"]

    def prepare(sample):
        if inp["dtype"] == np.int8:
            q = np.round(sample / in_scale + in_zero)
            return np.clip(q, -128, 127).astype(np.int8)
        return sample.astype(np.float32)

    interpreter.set_tensor(inp["index"], prepare(x[0:1]))
    interpreter.invoke()

    preds, times = [], []
    for i in range(len(x)):
        interpreter.set_tensor(inp["index"], prepare(x[i:i + 1]))
        start = time.perf_counter()
        interpreter.invoke()
        times.append(time.perf_counter() - start)
        preds.append(np.argmax(interpreter.get_tensor(out["index"])[0]))

    acc = np.mean(np.array(preds) == y) * 100
    latency_ms = np.mean(times) * 1000
    return acc, latency_ms, inp["dtype"].__name__, out["dtype"].__name__

## 8. Quantization-Aware Training (QAT)
`quantize_model` inserts fake-quantization layers into the trained baseline. The model is then fine-tuned for a few epochs with a very small learning rate.

In [15]:
qat_model = tfmot.quantization.keras.quantize_model(model)

qat_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss=LOSS, metrics=["accuracy"])

_, qat_before_acc = qat_model.evaluate(x_test, y_test, verbose=0)
print(f"QAT model accuracy before fine-tuning: {qat_before_acc * 100:.2f}%")

history_qat = qat_model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1, verbose=2)

_, keras_qat_acc = qat_model.evaluate(x_test, y_test, verbose=0)
print(f"QAT model accuracy after fine-tuning:  {keras_qat_acc * 100:.2f}%")

QAT model accuracy before fine-tuning: 10.90%
Epoch 1/5
71/71 - 15s - loss: 1.6042 - accuracy: 0.5316 - val_loss: 4.1095 - val_accuracy: 0.1500 - 15s/epoch - 214ms/step
Epoch 2/5
71/71 - 8s - loss: 1.2300 - accuracy: 0.6182 - val_loss: 2.9097 - val_accuracy: 0.2480 - 8s/epoch - 117ms/step
Epoch 3/5
71/71 - 8s - loss: 0.9178 - accuracy: 0.7049 - val_loss: 1.9373 - val_accuracy: 0.4100 - 8s/epoch - 115ms/step
Epoch 4/5
71/71 - 8s - loss: 0.6840 - accuracy: 0.7716 - val_loss: 1.3873 - val_accuracy: 0.5540 - 8s/epoch - 115ms/step
Epoch 5/5
71/71 - 9s - loss: 0.5107 - accuracy: 0.8251 - val_loss: 1.0248 - val_accuracy: 0.6700 - 9s/epoch - 120ms/step
QAT model accuracy after fine-tuning:  65.30%


## 9. Convert the QAT model to full INT8 TFLite
No `representative_dataset` is given here. The fake-quantization layers already learned the value ranges during training.

In [18]:
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
qat_tflite = converter.convert()
with open(QAT_PATH, "wb") as f:
    f.write(qat_tflite)
print("Saved", QAT_PATH, "(converted without a representative dataset)")

check = tf.lite.Interpreter(model_content=qat_tflite)
check.allocate_tensors()
dtypes = pd.Series([d["dtype"].__name__ for d in check.get_tensor_details()]).value_counts()
print("Tensor types in the QAT model:")
print(dtypes)

INFO:tensorflow:Assets written to: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmp14la04g5/assets


INFO:tensorflow:Assets written to: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmp14la04g5/assets
/Users/Tanmay/Downloads/tf_env/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1791004102.127361 4204729 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1791004102.127372 4204729 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.


Saved tflite_models/qat_int8.tflite (converted without a representative dataset)
Tensor types in the QAT model:
int8     118
int32     54
Name: count, dtype: int64


I0000 00:00:1791004102.127530 4204729 reader.cc:83] Reading SavedModel from: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmp14la04g5
I0000 00:00:1791004102.154951 4204729 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1791004102.154960 4204729 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmp14la04g5
I0000 00:00:1791004102.305487 4204729 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1791004102.944468 4204729 loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folders/_v/pk2x5_cn235c84mks_bl5r000000gp/T/tmp14la04g5
I0000 00:00:1791004103.133200 4204729 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 1005673 microseconds.
/Users/Tanmay/Downloads/tf_env/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use

## 10. Comparison table

In [25]:
rows = []
for name, path in [("Float32 baseline", FLOAT_PATH),
                   ("PTQ full INT8", PTQ_PATH),
                   ("QAT full INT8", QAT_PATH)]:
    acc, lat, in_type, out_type = evaluate_tflite(path, x_test, y_test)
    rows.append({
        "Model": name,
        "Size (KB)": round(os.path.getsize(path) / 1024, 2),
        "Test accuracy (%)": round(acc, 2),
        "Avg latency (ms)": round(lat, 3),
        "Input / output type": f"{in_type} / {out_type}",
    })

results = pd.DataFrame(rows)
display(results)

print(f"Keras float32 baseline accuracy: {keras_baseline_acc * 100:.2f}%")
print(f"Keras QAT model accuracy:        {keras_qat_acc * 100:.2f}%")
print("Latency measured on the local Mac CPU with 1 thread, averaged over", len(x_test), "test images.")

/Users/Tanmay/Downloads/tf_env/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,Model,Size (KB),Test accuracy (%),Avg latency (ms),Input / output type
0,Float32 baseline,1604.95,74.7,0.316,float32 / float32
1,PTQ full INT8,618.08,68.7,0.283,int8 / int8
2,QAT full INT8,624.15,64.5,0.283,int8 / int8


Keras float32 baseline accuracy: 74.70%
Keras QAT model accuracy:        65.30%
Latency measured on the local Mac CPU with 1 thread, averaged over 1000 test images.


## 11. Result summary (auto-filled from the table)

In [20]:
f_row, p_row, q_row = results.iloc[0], results.iloc[1], results.iloc[2]

print(f"Float32 baseline: {f_row['Test accuracy (%)']:.2f}% accuracy, {f_row['Size (KB)']:.2f} KB, {f_row['Avg latency (ms)']:.3f} ms")
print(f"PTQ full INT8:    {p_row['Test accuracy (%)']:.2f}% accuracy, {p_row['Size (KB)']:.2f} KB, {p_row['Avg latency (ms)']:.3f} ms")
print(f"QAT full INT8:    {q_row['Test accuracy (%)']:.2f}% accuracy, {q_row['Size (KB)']:.2f} KB, {q_row['Avg latency (ms)']:.3f} ms")
print()
print(f"Accuracy drop from float32, PTQ: {f_row['Test accuracy (%)'] - p_row['Test accuracy (%)']:.2f} percentage points")
print(f"Accuracy drop from float32, QAT: {f_row['Test accuracy (%)'] - q_row['Test accuracy (%)']:.2f} percentage points")
print(f"Size reduction from float32, PTQ: {f_row['Size (KB)'] / p_row['Size (KB)']:.2f}x")
print(f"Size reduction from float32, QAT: {f_row['Size (KB)'] / q_row['Size (KB)']:.2f}x")

Float32 baseline: 74.70% accuracy, 1604.95 KB, 0.317 ms
PTQ full INT8:    68.70% accuracy, 618.08 KB, 0.291 ms
QAT full INT8:    64.50% accuracy, 624.15 KB, 0.291 ms

Accuracy drop from float32, PTQ: 6.00 percentage points
Accuracy drop from float32, QAT: 10.20 percentage points
Size reduction from float32, PTQ: 2.60x
Size reduction from float32, QAT: 2.57x


## 12. Download the three .tflite models

In [23]:
shutil.make_archive("lab4_tflite_models", "zip", "tflite_models")
for f in sorted(os.listdir("tflite_models")):
    print(f, round(os.path.getsize(os.path.join("tflite_models", f)) / 1024, 2), "KB")

print("Zip saved at:", os.path.abspath("lab4_tflite_models.zip"))

baseline_float32.tflite 1604.95 KB
ptq_int8.tflite 618.08 KB
qat_int8.tflite 624.15 KB
Zip saved at: /Users/Tanmay/Downloads/lab4_tflite_models.zip


## Questions

**1. Why does Post-Training Quantization reduce model accuracy?**

PTQ converts weights and activations from 32-bit floating point to 8-bit integers after training is already finished. An 8-bit integer can hold only 256 levels, so every value is rounded to the nearest level (rounding error), and values outside the calibrated range are clipped (clipping error). The calibrated range comes from a small representative dataset, so it may not match all real inputs. The model never saw these errors during training, so it cannot adjust to them. The small errors add up from layer to layer and change the final output. Compact models such as MobileNetV2 with alpha = 0.35 are more sensitive because they have fewer redundant weights to absorb the error, and their depthwise convolution layers can have very different value ranges across channels.

**2. What does QAT simulate during training, and how does this help the model?**

QAT inserts fake-quantization nodes into the model. In the forward pass, these nodes round weights and activations to INT8 levels and then convert them back to float, so the model sees the same rounding and clipping it will face after INT8 conversion. In the backward pass, gradients pass through the rounding step as if it were not there (the Straight-Through Estimator, STE), and the weights are still updated in float. This helps in two ways: the model learns weights that still work well after rounding, and the fake-quantization nodes learn suitable minimum and maximum ranges for each activation tensor.

**3. Was a representative dataset required to convert the QAT model? Justify your answer.**

No. The QAT model was converted to full INT8 without a `representative_dataset` (Section 9). The fake-quantization layers already store the minimum and maximum range for each tensor, learned during training. The converter uses these stored ranges to compute the scale and zero point for every tensor, so it does not need to run calibration data through the model. PTQ needed a representative dataset because the float baseline had no range information.

**4. Did QAT change the size of the INT8 model? Explain.**

No, the size stayed about the same (see the Size column in the table). Both the PTQ and QAT models have the same layers and the same number of parameters, and both store weights as 8-bit integers. QAT only changes the values of the weights and the quantization ranges, not how many values are stored. Any small difference in kilobytes comes from extra metadata or quantization parameters, not from the model itself. Both INT8 models are roughly 3 to 4 times smaller than the float32 model.

**5. Under what conditions is PTQ sufficient, so that QAT is not needed?**

PTQ is usually enough when:
- the accuracy drop after PTQ is small enough for the application (for example, under about 1 percentage point),
- the model is large or has many redundant parameters, so it tolerates rounding well,
- a good representative dataset that covers the real input range is available,
- per-channel weight quantization is used, which reduces error in convolution layers,
- there is no time, training data or training pipeline available to retrain the model.

QAT is worth the extra training when the model is small and compact, when some layers are very sensitive to quantization, or when the PTQ accuracy drop is not acceptable.

## Result

The float32 baseline reached 74.70% test accuracy with a size of 1604.95 KB. Full INT8 PTQ reduced the size to 618.08 KB (2.60x smaller) with 68.70% accuracy, a drop of 6.00 percentage points. QAT produced a full INT8 model of 624.15 KB (2.57x smaller) with 64.50% accuracy, a drop of 10.20 percentage points from the baseline. In this experiment, QAT kept the same size reduction as PTQ but did not recover accuracy, scoring 4.20 percentage points lower than PTQ. The Keras QAT model also reached only 65.30% before conversion, so the accuracy loss came from the QAT fine-tuning stage and not from the TFLite conversion. A likely reason is that 5 epochs of QAT fine-tuning on 4,500 training images were not enough for the compact MobileNetV2 (alpha = 0.35) to adapt to the simulated quantization. Average latency was 0.317 ms for float32, 0.291 ms for PTQ and 0.291 ms for QAT on a local Mac CPU, which may not match the latency on a real microcontroller or edge device. For this setup, PTQ gave the better result.